In [ ]:
import numpy as np
import numpy.polynomial.polynomial as nppol
import glob
from tqdm import tqdm
from itertools import repeat
from datetime import datetime
from h5py import File
import multiprocessing
import os 
from scipy.io import readsav
from astropy.time import Time

from spacepy import pycdf
from matplotlib import pyplot as plt
import matplotlib.colors as colors
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from mpl_toolkits import axes_grid1
import matplotlib.gridspec as gridspec

from scipy.interpolate import interp1d

import pandas as pd
from scipy.stats import skewnorm
import scipy as sp


In [40]:
from tfcat import TFCat
from tfcat.validate import JSONSCHEMA_URI

from tfcat_tools.edit import merge
from tfcat_tools.mask import Mask
from tfcat_tools.featureaddons import FeatureCollection

from tfcat_tools.read_juno_ephemeris_from_amda import *
from tfcat_tools.jupiter_ephem import *
from tfcat_tools.utils import *
from tfcat_tools.handling_hdf5 import *

import importlib
import paths

Path to data, catalogue and ephemeris files

In [ ]:
%load_ext autoreload
%autoreload 2

importlib.reload(paths)

#print("Workspace:", paths.WORKSPACE)
#print("Data root:", paths.DATA_ROOT)
for name in ["DATA_ROOT", "INPUT_DATA", 
             "OUTPUTS", "EPHEM", "MMESH", "TFCAT",
             ]:
    path = getattr(paths, name)
#    print(f"{name:20s} {path.exists()}  {path}")

Workspace: /obs/kkesri
Data root: /obs/kkesri/data_and_outputs
DATA_ROOT            True  /obs/kkesri/data_and_outputs
INPUT_DATA           True  /obs/kkesri/data_and_outputs/spacecraft_and_model
OUTPUTS              True  /obs/kkesri/data_and_outputs/datasets_ERICA-JS
EPHEM                True  /obs/kkesri/data_and_outputs/ephem
MMESH                True  /obs/kkesri/data_and_outputs/spacecraft_and_model/MMESH_data
TFCAT                True  /obs/kkesri/data_and_outputs/spacecraft_and_model/TFCat_catalog_2.1


In [13]:
# ==========================================
# EPHEMERIS FILES
# ==========================================

data_path_IAU_ephemeris = paths.EPHEM / "juno_jup_xyz_iau_2016_2025.txt"
data_path_JSO_ephemeris = paths.EPHEM / "juno_jup_xyz_jso_2016_2025.txt"

# ==========================================
# TFCAT CATALOGUES
# ==========================================

YEARS = range(2016, 2022)

path_cat = {str(year): paths.TFCAT / f"juno_waves_lira_tfcat_{year}.json" for year in YEARS}
path_dir_data  = '/obs/juno_data_preprocessed/'

Definition of output path and keywords

In [6]:
directory_path_out = paths.TFCAT
savefig = True

Constants Definition

In [7]:
AU = 149597870.700 # 1 astronomical unit in km
RJ = 71492. # Jovian radius in km

Loading Ephemeris

In [8]:
all_ephemeris = juno_ephemeris_from_amda(data_path_IAU_ephemeris)
local_time_ephemeris = juno_ephemeris_from_amda(data_path_JSO_ephemeris)

#print(threshold)

Time range definition

In [ ]:
import datetime

time_start = datetime.datetime(2016,6,1)
time_end = datetime.datetime(2021, 3, 1, 0, 0, 0) # or After datetime.datetime(2021, 9, 1, 0, 0, 0). Gap in between

HDF5 data files reading

In [14]:
hdf5_files_2016 = ["jno_wav_cdr_lesia_2016_apr_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2016_sep_dec.preprocessed.hdf5"]
hdf5_files_2017 = ["jno_wav_cdr_lesia_2017_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2017_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2017_sep_dec.preprocessed.hdf5"]
hdf5_files_2018 = ["jno_wav_cdr_lesia_2018_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2018_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2018_sep_dec.preprocessed.hdf5"]
hdf5_files_2019 = ["jno_wav_cdr_lesia_2019_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2019_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2019_sep_dec.preprocessed.hdf5"]
hdf5_files_2020 = ["jno_wav_cdr_lesia_2020_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2020_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2020_sep_dec.preprocessed.hdf5"]
hdf5_files_2021 = ["jno_wav_cdr_lesia_2021_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2021_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2021_sep_dec.preprocessed.hdf5"]
hdf5_files_2022 = ["jno_wav_cdr_lesia_with_fce_2022_jan_apr.preprocessed.hdf5",
                   "jno_wav_cdr_lesia_2022_may_aug.preprocessed.hdf5",
                   "jno_wav_cdr_lesia_2022_sep_dec.preprocessed.hdf5"]
hdf5_files_2023 = ["jno_wav_cdr_lesia_2023_jan_apr.preprocessed.hdf5"]


# /!\ be careful, do not take into account beg. of 2021. bKOM is inverse mask, so if catalogue not done, mask takes everything
# /!\ do not take data at PJ, so wh en d < 2 R_Jupiter

# Dictionary to hold file lists for each year
files_dict = {
    2016: hdf5_files_2016,
    2017: hdf5_files_2017,
    2018: hdf5_files_2018,
    2019: hdf5_files_2019,
    2020: hdf5_files_2020,
    2021: hdf5_files_2021,
    #2022: hdf5_files_2022,
    #2023: hdf5_files_2023,
}


# Variables to concatenate
variables = ['Flux', 'Time']

# Dictionary to store concatenated data for each year
yearly_data = {}

# Process each year
for iyear, files in files_dict.items():
    yearly_data[iyear] = concatenate_hdf5_files(files, variables, path_files = path_dir_data)


Frequency reading

In [15]:
with File(path_dir_data+hdf5_files_2016[0]) as hdf_file:
    Frequency = numpy.array(hdf_file['Frequency'])

Data variables and TFCat catalogues sorting and creation of dictionnaries

In [20]:
file_name_hdf5 = {f'{iyear}': f'mask_{iyear}.hdf5' for iyear in YEARS}
cat = {f'{iyear}': sort_cat(TFCat.from_file(path_cat[f'{iyear}'])) for iyear in YEARS}
for iyear in YEARS:
    cat[f'{iyear}'].properties['instrument_name'] = "Waves_" 

time = {f'{iyear}': numpy.array(Time(yearly_data[iyear]['Time'], format="jd").unix) for iyear in YEARS}
data = {f'{iyear}': numpy.array(yearly_data[iyear]['Flux']) for iyear in YEARS}

Selection in time

In [21]:
mask_time_start = time[time_start.strftime("%Y")] >= Time(time_start,format="datetime").unix
time[time_start.strftime("%Y")] = time[time_start.strftime("%Y")][mask_time_start]
data[time_start.strftime("%Y")] = data[time_start.strftime("%Y")][mask_time_start,:]

mask_time_end = time[time_end.strftime("%Y")] <= Time(time_end,format="datetime").unix
time[time_end.strftime("%Y")] = time[time_end.strftime("%Y")][mask_time_end]
data[time_end.strftime("%Y")] = data[time_end.strftime("%Y")][mask_time_end,:]

Ephemerides calculation only on the wanted time range

In [22]:
time_arrays = list(time.values())
time_over_year_sec = numpy.concatenate(time_arrays)
# Normalization at 1 AU
mask_time = (all_ephemeris[0]>= time_start) & (all_ephemeris[0]< time_end)
ephem_x = all_ephemeris[1][mask_time]
ephem_y = all_ephemeris[2][mask_time]
ephem_z = all_ephemeris[3][mask_time]
ephem_date = all_ephemeris[0][mask_time]

distance = numpy.sqrt(ephem_x**2 + ephem_y**2 + ephem_z**2)
lat = (numpy.pi-numpy.arccos(ephem_z / distance)) * 180/numpy.pi
lon = numpy.pi-numpy.arctan2(ephem_y,ephem_x) * 180/numpy.pi

distance = numpy.sqrt(ephem_x**2 + ephem_y**2 + ephem_z**2)
lat = 90 - (numpy.arccos(ephem_z / distance))*180/numpy.pi 
lon = (360 - ((numpy.arctan2(ephem_y,ephem_x))*180/numpy.pi)) % 360

time_ephem_sec = Time(ephem_date, format='datetime').unix
oblon=numpy.interp(time_over_year_sec,time_ephem_sec,lon)
oblat=numpy.interp(time_over_year_sec,time_ephem_sec,lat) 
obdist=numpy.interp(time_over_year_sec,time_ephem_sec,distance)
obmlat=10.31*numpy.cos((196.61-oblon)*numpy.pi/180.)+oblat

In [23]:
print(Time(time_over_year_sec[0], format = "unix").isot, Time(time_over_year_sec[-1], format = "unix").isot,
        Time(time_ephem_sec[0], format = "unix").isot, Time(time_ephem_sec[-1], format = "unix").isot)

2016-06-01T00:00:08.184 2021-02-28T23:59:09.184 2016-06-01T00:05:00.000 2021-02-28T23:55:00.000


Frequency range selection

In [24]:
#time_index = numpy.where((time.datetime >= time_start) & (time.datetime < time_end))[0]

index_freq_LFR = numpy.where((Frequency > 1) & (Frequency < 142))[0][0:-1]
index_freq_bKOM = numpy.where((Frequency > 10) & (Frequency < 142))[0][0:-1] # index_freq high (>70) and low (<70)
index_freq_DAM = numpy.where(Frequency >3000)[0]

Calculating or loading normalized_intensity_bKOM

In [25]:
calculate_intensity = False
if calculate_intensity:
    intensity_bKOM_list = []

    for index, iyear in enumerate(year):
        print(f"{iyear}")
        mask_juno_bKOM = Mask(
                            cat[f'{iyear}'],
                            time[f'{iyear}'],
                            Frequency[index_freq_bKOM],
                            data[f'{iyear}'][:,index_freq_bKOM]
                            )

        mask_juno_bKOM_data =  mask_juno_bKOM.apply('QP','nLF','nKOM','QP?','nLF?','nKOM?','anom',invert=True)
        mask_juno_bKOM_data = mask_juno_bKOM_data * 377
        #integrated_intensity_bKOM = numpy.sum(mask_juno_bKOM_data, axis=1)

        intensity_bKOM_list.append(mask_juno_bKOM_data)

        #if index == 0:
        #    integrated_intensity_bKOM_over_year = integrated_intensity_bKOM
        #else:
        #    integrated_intensity_bKOM_over_year = [integrated_intensity_bKOM_over_year, integrated_intensity_bKOM]
    intensity_bKOM_over_year = numpy.concatenate(intensity_bKOM_list)

    #integrated_intensity_bKOM_over_year_flattened_arrays = flatten(integrated_intensity_bKOM_over_year)
    #normalized_integrated_intensity_bKOM = numpy.concatenate(integrated_intensity_bKOM_over_year_flattened_arrays)*(obdist/AU)**2
    normalized_intensity_bKOM = intensity_bKOM_over_year * (obdist[:, numpy.newaxis]*RJ / AU)**2    

    output_file = File('normalized_intensity_bKOM_catalog_v21.hdf5', 'w')
    output_file.attrs.create('observer', 'Juno')
    output_file.create_dataset('normalized_intensity_bKOM', data=normalized_intensity_bKOM)
    output_file['normalized_intensity_bKOM'].attrs.create('units', 'W/m2/Hz@1AU')
    output_file.close()
else:
    normalized_intensity_bKOM = load_intensity_from_hdf5_file("/obs/kkesri/data_and_outputs/spacecraft_and_model/normalized_intensity_bKOM_catalog_v20.hdf5")


Filtering with distance

In [26]:
mask_distance = obdist < 10
normalized_intensity_bKOM[mask_distance,:] = numpy.nan

Calculating background from normalized_intensity_bKOM data

In [27]:
background_from_data = numpy.zeros(normalized_intensity_bKOM.shape[1])
for ifreq in range(normalized_intensity_bKOM.shape[1]):
    mask = normalized_intensity_bKOM[:,ifreq] > 0
    background_from_data[ifreq] =  numpy.nanmean(normalized_intensity_bKOM[mask, ifreq])
    

Calculating integrated intensity over frequency

In [28]:
normalized_integrated_intensity_bKOM = numpy.sum(normalized_intensity_bKOM, axis=1)

Calculating integrated occurrence over frequency using background data

In [29]:
occurrence = thresholding_data(normalized_intensity_bKOM, background_from_data)
integrated_occurrence = numpy.sum(occurrence, axis = 1)
mask = integrated_occurrence > 0
integrated_occurrence[mask] = 1

In [30]:
time_datetime = pd.to_datetime(time_over_year_sec, unit='s')